# 04 — Fine-tuning mBERT

`bert-base-multilingual-cased` con pérdida ponderada y early stopping.

## Dataset B2 (informe de verificación)

- Train: **3,293** filas (`data/processed/train.csv`), columnas `origen` / `plantilla_id`.
- Val/test: **459 / 460**, idénticos al v1 (comparables).
- Pesos de clase B2 en `data/processed/metadata.json` (no recalcular tras filtrar solapes).
- Entrenamiento recomendado (sin `MAX_STEPS`):

```bash
python scripts/train_mbert.py --b2 --exclude-adversarial-overlap --output-dir models/mbert-sv-b2
```

- Amenazas sintéticas: masa efectiva ~90 en la pérdida (268 filas × peso ≈ 0.336).
- Label smoothing: 0.05. Checkpoint con `id2label` real al guardar.

In [ ]:
import sys
from pathlib import Path

_bootstrap = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(_bootstrap / 'src') not in sys.path:
    sys.path.insert(0, str(_bootstrap / 'src'))

from discurso_odio.paths import PROJECT_ROOT

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from discurso_odio.config import load_settings
from discurso_odio.data.loaders import load_all_splits, load_split, get_class_weights
from discurso_odio.paths import REPORTS_FIGURES, REPORTS_METRICS
from discurso_odio.utils.plotting import setup_plot_style, get_label_names, get_class_palette
from discurso_odio.utils.seed import set_seed

settings = load_settings()
set_seed(settings['project']['seed'])
setup_plot_style()
LABELS = get_label_names()
REPORTS_FIGURES.mkdir(parents=True, exist_ok=True)
REPORTS_METRICS.mkdir(parents=True, exist_ok=True)
print('Proyecto:', settings['project']['name'])

from discurso_odio.models.mbert import train_mbert, get_device
from discurso_odio.paths import MODELS_DIR



In [ ]:
print('Dispositivo:', get_device())
train = load_split('train')
val = load_split('val')
print(f'Train: {len(train)} | Val: {len(val)}')


## Entrenamiento principal (texto_modelo)


In [ ]:
model_path = MODELS_DIR / 'mbert-sv'
if model_path.exists() and (model_path / 'config.json').exists():
    print('Modelo ya entrenado en', model_path, '- omitiendo re-entrenamiento.')
    print('Para re-entrenar, elimine el directorio models/mbert-sv/')
else:
    trainer = train_mbert(train, val, output_dir=model_path, text_col='texto_modelo')
    print('Métricas validación:', trainer.evaluate())


## Ablación: texto_agresivo (grupo de control)


In [ ]:
# Descomentar para ejecutar ablación completa (~30-60 min adicionales)
# trainer_ag = train_mbert(train, val, output_dir=MODELS_DIR / 'mbert-sv-agresivo', text_col='texto_agresivo')
# print('Ablación agresivo:', trainer_ag.evaluate())
print('Ablación documentada. Ejecutar celdas comentadas para comparar variantes.')
